# 09 · Acesso à Energia Elétrica — 01_ETL
Amazonas (AM) · EPSG:31980 · **Fonte: Censo Demográfico 2010 (Resultados do Universo, por setor censitário)**

**Etapas**
1. Download dos resultados do Censo 2010 para o AM
2. Reprojeção para EPSG:31980 e join com os municípios do AM
3. Cálculo da % de domicílios sem energia por zona (rural/urbana)
4. Exportação de `energia-am.geojson` e `municipios-am.geojson`

**Entrada local:** `dados/geojs-13-mun.json` (malha municipal do AM). Os CSVs do IBGE são baixados na etapa 1.

**Variáveis (Domicilio01):** `V002` domicílios particulares permanentes (denominador) · `V043` com energia · `V046` **sem energia**.

> Os dados são de **2010**: não refletem os avanços posteriores (ex.: Luz para Todos).

In [28]:
# @title
from pathlib import Path
import re, shutil, zipfile
from urllib.request import urlopen, Request
from urllib.error import URLError

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# --- Parâmetros ---
CRS_PROJ   = "EPSG:31980"   # SIRGAS 2000 / UTM 20S: usado nos cálculos (metros)
CRS_EXPORT = "EPSG:4326"    # CRS dos GeoJSON exportados (lon/lat): é o que mapas e o padrão GeoJSON esperam
COD_MANAUS = 1302603
SEDE_MANAUS = (-60.0217, -3.1190)   # (lon, lat) da cidade de Manaus

DIR_DADOS = Path("dados")
DIR_SAIDA = Path("saida")
DIR_DADOS.mkdir(exist_ok=True); DIR_SAIDA.mkdir(exist_ok=True)

ARQ_MUN    = DIR_DADOS / "geojs-13-mun.json"
ARQ_DOM01  = DIR_DADOS / "Domicilio01_AM.csv"
ARQ_BASICO = DIR_DADOS / "Basico_AM.csv"

URL_BASE = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2010/Resultados_do_Universo/Agregados_por_Setores_Censitarios/"
ARQUIVOS_CENSO = ["Domicilio01_AM.csv", "Basico_AM.csv"]
REFAZER_DOWNLOAD = False

CSV_KW = dict(sep=";", encoding="latin1", dtype=str, na_values=["x", "X", ".", ""])

In [29]:
URL_MALHA_MUN = "https://raw.githubusercontent.com/tbrugz/geodata-br/refs/heads/master/geojson/geojs-13-mun.json"

## 1. Download dos resultados do Censo 2010 (AM)
Descobre na listagem do IBGE o zip mais recente do AM (o nome muda a cada republicação, ex.: `AM_20260615.zip`), baixa para `dados/` e extrai `Domicilio01_AM.csv` e `Basico_AM.csv`. Se os CSVs já existirem, não baixa de novo (`REFAZER_DOWNLOAD = True` força).

In [30]:
# @title
def _abrir(url):
    return urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=120)

def baixar_censo_am():
    if not REFAZER_DOWNLOAD and all((DIR_DADOS / a).exists() for a in ARQUIVOS_CENSO):
        print("CSVs já existem em", DIR_DADOS.resolve(), "- nada a baixar.")
        return
    try:
        html = _abrir(URL_BASE).read().decode("utf-8", "ignore")
        zips = sorted(set(re.findall(r'href="(AM_\d{8}\.zip)"', html)))
        if not zips:
            raise RuntimeError("Nenhum AM_AAAAMMDD.zip encontrado na listagem do IBGE.")
        nome = zips[-1]                                   # data mais recente
        destino = DIR_DADOS / nome
        if not destino.exists():
            print(f"Baixando {nome} (~31 MB)...")
            parcial = destino.with_suffix(".part")
            with _abrir(URL_BASE + nome) as r, open(parcial, "wb") as f:
                shutil.copyfileobj(r, f, 1024 * 1024)
            parcial.rename(destino)
    except (URLError, OSError, RuntimeError) as e:
        raise SystemExit(f"Falha no download ({e}). Baixe manualmente em {URL_BASE} "
                         f"e extraia {ARQUIVOS_CENSO} em '{DIR_DADOS}/'.")

    with zipfile.ZipFile(destino) as z:
        membros = {Path(n).name.lower(): n for n in z.namelist()}
        for a in ARQUIVOS_CENSO:
            if a.lower() not in membros:
                raise FileNotFoundError(f"{a} não está em {nome}. Conteúdo: {z.namelist()[:15]}")
            with z.open(membros[a.lower()]) as src, open(DIR_DADOS / a, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print("extraído:", DIR_DADOS / a)

baixar_censo_am()

CSVs já existem em /content/dados - nada a baixar.


In [31]:
# @title
def baixar_malha_municipal_am():
    if not REFAZER_DOWNLOAD and ARQ_MUN.exists():
        print("Malha municipal já existe em", ARQ_MUN.resolve(), "- nada a baixar.")
        return
    try:
        print(f"Baixando {ARQ_MUN.name}...")
        parcial = ARQ_MUN.with_suffix(".part")
        with _abrir(URL_MALHA_MUN) as r, open(parcial, "wb") as f:
            shutil.copyfileobj(r, f, 1024 * 1024)
        parcial.rename(ARQ_MUN)
    except (URLError, OSError, RuntimeError) as e:
        raise SystemExit(f"Falha no download da malha municipal ({e}). Baixe manualmente em {URL_MALHA_MUN} "
                         f"e salve como '{ARQ_MUN.name}' em '{DIR_DADOS}/'.")


In [32]:
baixar_malha_municipal_am()

Malha municipal já existe em /content/dados/geojs-13-mun.json - nada a baixar.


## 2. Reprojeção para EPSG:31980 e join com os municípios do AM

### 2.1 Malha municipal reprojetada

In [33]:
mun = gpd.read_file(ARQ_MUN)
print("colunas:", mun.columns.tolist(), "| CRS original:", mun.crs)

mun = mun.rename(columns={"id": "cd_mun", "name": "nm_mun"})[["cd_mun", "nm_mun", "geometry"]]
mun["cd_mun"] = mun["cd_mun"].astype(int)
if mun.crs is None:
    mun = mun.set_crs("EPSG:4326")
mun = mun.to_crs(CRS_PROJ)                                  # <- reprojeção

assert len(mun) == 62, f"AM tem 62 municípios; encontrei {len(mun)}"
assert mun["cd_mun"].is_unique and COD_MANAUS in set(mun["cd_mun"])
print("CRS após reprojeção:", mun.crs.to_string())

colunas: ['id', 'name', 'description', 'geometry'] | CRS original: EPSG:4326
CRS após reprojeção: EPSG:31980


### 2.2 Setores censitários (Censo 2010)

In [34]:
# @title
def ler_csv(caminho):
    d = pd.read_csv(caminho, **CSV_KW)
    d.columns = d.columns.str.strip().str.lower()
    return d

raw = ler_csv(ARQ_DOM01)
col_sit = "situacao_setor" if "situacao_setor" in raw.columns else "situacao"

setor = raw["cod_setor"].str.strip()
assert setor.str.fullmatch(r"\d{15}").all(), \
    "Cod_setor fora do padrão de 15 dígitos (CSV antigo/corrompido?). Baixe a versão republicada pelo IBGE."

df = pd.DataFrame({
    "cd_mun":    setor.str[:7].astype(int),                         # UF(2) + município(5)
    "sit":       pd.to_numeric(raw[col_sit], errors="coerce"),      # 1-3 urbana | 4-8 rural
    "dom_v001":  pd.to_numeric(raw["v001"], errors="coerce"),       # domicílios (particulares + coletivos)
    "dom_total": pd.to_numeric(raw["v002"], errors="coerce"),       # particulares permanentes
    "dom_com":   pd.to_numeric(raw["v043"], errors="coerce"),
    "dom_sem":   pd.to_numeric(raw["v046"], errors="coerce"),
})
df = df[df["cd_mun"] // 100000 == 13].copy()                        # só AM
df["zona"] = df["sit"].map(lambda v: None if pd.isna(v) else ("urbana" if v <= 3 else "rural"))
print(f"Setores AM: {len(df)} | municípios: {df['cd_mun'].nunique()}")

Setores AM: 5641 | municípios: 62


### 2.3 Qualidade dos dados
O IBGE omite (`x`) as variáveis dos setores com menos de 5 domicílios particulares permanentes. Esses setores ficam fora do cálculo; abaixo, o tamanho do viés.

In [35]:
# @title
omitido = df[["dom_total", "dom_sem", "zona"]].isna().any(axis=1)
print(f"Setores omitidos: {omitido.sum()} de {len(df)} ({omitido.mean():.1%})")
if df["dom_v001"].notna().any():
    dom_omit = df.loc[omitido, "dom_v001"].sum()
    print(f"Domicílios (V001) nesses setores: {dom_omit:.0f} ({dom_omit / df['dom_v001'].sum():.2%} do total do AM)")
df = df[~omitido].copy()

dif = (df["dom_com"] + df["dom_sem"] - df["dom_total"]).abs()
print("Setores em que com+sem != total:", int((dif > 0).sum()))
assert ((df["dom_sem"] >= 0) & (df["dom_sem"] <= df["dom_total"])).all(), "dom_sem fora de [0, total]"

Setores omitidos: 205 de 5641 (3.6%)
Domicílios (V001) nesses setores: 3073 (0.38% do total do AM)
Setores em que com+sem != total: 0


### 2.4 Agregação por município × zona e join
Somam-se **contagens** por município e zona; os percentuais só são calculados depois (etapa 3), nunca como média de percentuais de setores.

In [36]:
# @title
cont = (df.groupby(["cd_mun", "zona"])[["dom_total", "dom_sem"]].sum()
          .unstack("zona").fillna(0).astype(int))
cont.columns = [f"{v}_{z}" for v, z in cont.columns]
for c in ["dom_total_rural", "dom_total_urbana", "dom_sem_rural", "dom_sem_urbana"]:
    if c not in cont:
        cont[c] = 0
cont["dom_total"] = cont["dom_total_rural"] + cont["dom_total_urbana"]
cont["dom_sem"]   = cont["dom_sem_rural"] + cont["dom_sem_urbana"]
cont = cont.reset_index()

# Mesorregião (divisão de 2010) e população, via Basico_AM.csv
bas = ler_csv(ARQ_BASICO)
bas["cd_mun"]  = pd.to_numeric(bas["cod_municipio"], errors="coerce")
bas["pop_dpp"] = pd.to_numeric(bas["v002"], errors="coerce")    # moradores em domicílios particulares permanentes
extra = (bas.dropna(subset=["cd_mun"]).assign(cd_mun=lambda d: d["cd_mun"].astype(int))
            .groupby("cd_mun")
            .agg(cd_meso=("cod_meso", "first"), nm_meso=("nome_da_meso", "first"), pop_dpp=("pop_dpp", "sum"))
            .reset_index())

# Join com a malha reprojetada
gdf = (mun.merge(cont, on="cd_mun", how="left", validate="1:1")
          .merge(extra, on="cd_mun", how="left", validate="1:1"))
sem_dados = gdf["dom_total"].isna().sum()
print(f"Municípios sem dados do Censo após o join: {sem_dados}")
assert sem_dados == 0, "Há municípios da malha sem correspondência no Censo (confira os códigos)."
gdf.drop(columns="geometry").head()

Municípios sem dados do Censo após o join: 0


,cd_mun,nm_mun,dom_total_rural,dom_total_urbana,dom_sem_rural,dom_sem_urbana,dom_total,dom_sem,cd_meso,nm_meso,pop_dpp
0,1300029,Alvarães,1118,1363,133,37,2481,170,1303,Centro Amazonense,14049.0
1,1300060,Amaturá,703,944,365,12,1647,377,1302,Sudoeste Amazonense,9368.0
2,1300086,Anamã,1061,806,193,4,1867,197,1303,Centro Amazonense,10186.0
3,1300102,Anori,1007,2053,264,15,3060,279,1303,Centro Amazonense,16257.0
4,1300144,Apuí,1879,2715,910,34,4594,944,1304,Sul Amazonense,17946.0


## 3. Cálculo da % de domicílios sem energia por zona (rural/urbana)

In [37]:
# @title
def pct(sem, total):
    return np.where(total > 0, 100 * sem / total, np.nan)      # zona sem domicílios -> NaN (não 0%)

gdf["pct_sem"]        = pct(gdf["dom_sem"], gdf["dom_total"])
gdf["pct_sem_rural"]  = pct(gdf["dom_sem_rural"], gdf["dom_total_rural"])
gdf["pct_sem_urbana"] = pct(gdf["dom_sem_urbana"], gdf["dom_total_urbana"])

# Distância a Manaus: centroide do município -> sede de Manaus (em metros no CRS projetado, depois km).
# É distância em linha reta, não fluvial. Manaus = 0 por definição.
sede = gpd.GeoSeries([Point(*SEDE_MANAUS)], crs="EPSG:4326").to_crs(CRS_PROJ).iloc[0]
gdf["dist_manaus_km"] = gdf.geometry.centroid.distance(sede) / 1000
gdf.loc[gdf["cd_mun"] == COD_MANAUS, "dist_manaus_km"] = 0.0

# Totais do estado por zona (referência para a análise rural × urbano)
estado = pd.DataFrame({
    "dom_total": [gdf["dom_total_rural"].sum(), gdf["dom_total_urbana"].sum(), gdf["dom_total"].sum()],
    "dom_sem":   [gdf["dom_sem_rural"].sum(),   gdf["dom_sem_urbana"].sum(),   gdf["dom_sem"].sum()],
}, index=["rural", "urbana", "total"])
estado["pct_sem"] = 100 * estado["dom_sem"] / estado["dom_total"]
display(estado.round(2))

gdf[["nm_mun", "dom_total", "pct_sem", "pct_sem_rural", "pct_sem_urbana", "dist_manaus_km"]] \
    .sort_values("pct_sem", ascending=False).head(10).round(2)

,dom_total,dom_sem,pct_sem
rural,140422,48488,34.53
urbana,657830,4442,0.68
total,798252,52930,6.63


,nm_mun,dom_total,pct_sem,pct_sem_rural,pct_sem_urbana,dist_manaus_km
5,Atalaia do Norte,2865,38.36,64.73,4.24,1335.78
49,Santa Isabel do Rio Negro,2603,34.04,62.10,1.98,685.87
51,São Gabriel da Cachoeira,7476,31.80,64.20,4.16,968.78
24,Guajará,2639,30.88,67.82,3.31,1497.51
29,Itamarati,1570,30.64,68.72,2.65,980.86
7,Barcelos,4408,28.68,59.36,1.16,479.87
52,São Paulo de Olivença,5325,28.54,50.33,3.13,1059.49
26,Ipixuna,3721,27.60,48.80,2.02,1343.71
42,Nova Olinda do Norte,5570,27.41,54.52,1.27,185.09
40,Maués,9985,26.35,52.35,1.70,300.42


## 4. Exportação: `energia-am.geojson` e `municipios-am.geojson`
Os cálculos usam EPSG:31980 (metros), mas os arquivos são gravados em **`CRS_EXPORT` = EPSG:4326 (lon/lat)**: coordenadas em metros aparecem em branco em geojson.io, Leaflet, GitHub etc. Nos notebooks 02 e 03, reprojete com `gdf.to_crs("EPSG:31980")` quando precisar de distâncias em metros.

In [38]:
# @title
cols_mun = ["cd_mun", "nm_mun", "cd_meso", "nm_meso", "pop_dpp", "dist_manaus_km"]
municipios_am = gdf[cols_mun + ["geometry"]].to_crs(CRS_EXPORT)
energia_am    = gdf.to_crs(CRS_EXPORT)

municipios_am.to_file(DIR_SAIDA / "municipios-am.geojson", driver="GeoJSON")
energia_am.to_file(DIR_SAIDA / "energia-am.geojson", driver="GeoJSON")

# Conferência: o AM fica aproximadamente entre lon -73,8 e -56,1 e lat -9,8 e 2,3
for nome in ["municipios-am", "energia-am"]:
    g = gpd.read_file(DIR_SAIDA / f"{nome}.geojson")
    minx, miny, maxx, maxy = g.total_bounds
    print(f"{nome}: {g.shape[0]} feições, {g.shape[1]} colunas | {g.crs.to_string()} | "
          f"lon {minx:.1f}..{maxx:.1f}, lat {miny:.1f}..{maxy:.1f}")
    assert len(g) == 62
    assert -75 < minx < maxx < -55 and -11 < miny < maxy < 3, "Coordenadas fora do AM: confira o CRS"

municipios-am: 62 feições, 7 colunas | EPSG:4326 | lon -73.8..-56.1, lat -9.8..2.2
energia-am: 62 feições, 16 colunas | EPSG:4326 | lon -73.8..-56.1, lat -9.8..2.2


## 5. Exportar arquivos para o Google Drive

Para enviar os arquivos gerados para o seu Google Drive, vamos primeiro montar o Drive no ambiente Colab e depois copiar os arquivos para o diretório desejado. Por padrão, os arquivos serão copiados para a raiz do seu Google Drive.

In [39]:
from google.colab import drive
import shutil

# Monta o Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# Define o diretório de destino no Google Drive (raiz do Drive)
drive_destination_path = Path('/content/drive/MyDrive/saida')

# Cria o diretório de destino se não existir
drive_destination_path.mkdir(parents=True, exist_ok=True)

# Copia os arquivos
files_to_copy = [
    DIR_SAIDA / "municipios-am.geojson",
    DIR_SAIDA / "energia-am.geojson"
]

for file_path in files_to_copy:
    if file_path.exists():
        shutil.copy(file_path, drive_destination_path)
        print(f"Arquivo '{file_path.name}' copiado para '{drive_destination_path}'")
    else:
        print(f"Erro: Arquivo '{file_path.name}' não encontrado em '{DIR_SAIDA}'")

Arquivo 'municipios-am.geojson' copiado para '/content/drive/MyDrive/saida'
Arquivo 'energia-am.geojson' copiado para '/content/drive/MyDrive/saida'
